In [1]:
import pandas as pd
from pathlib import Path

data_path = Path("../data/raw/bengaluru/bus")

routes = pd.read_csv(data_path / "routes.txt", dtype=str)
trips = pd.read_csv(data_path / "trips.txt", dtype=str)
stops = pd.read_csv(data_path / "stops.txt", dtype=str)
stop_times = pd.read_csv(data_path / "stop_times.txt", dtype=str)

print("Routes:", routes.shape)
print("Trips:", trips.shape)
print("Stops:", stops.shape)
print("Stop times:", stop_times.shape)

print("\nStop times columns:", stop_times.columns.tolist())

Routes: (4434, 5)
Trips: (57836, 6)
Stops: (9960, 9)
Stop times: (1540686, 6)

Stop times columns: ['trip_id', 'arrival_time', 'departure_time', 'stop_id', 'stop_sequence', 'stop_headsign']


In [2]:
# Count distinct trips per route
trip_counts = trips.groupby("route_id")["trip_id"].nunique().reset_index(name="total_trips")

# Count distinct stops per route
trip_stops = trips[["route_id", "trip_id"]].merge(
    stop_times[["trip_id", "stop_id"]],
    on="trip_id",
    how="left"
)

stop_counts = trip_stops.groupby("route_id")["stop_id"].nunique().reset_index(name="total_stops")

# Combine features
route_features = routes.merge(trip_counts, on="route_id", how="left")
route_features = route_features.merge(stop_counts, on="route_id", how="left")

print(route_features.head())

                                     route_long_name route_short_name  \
0  2nd Stage 9th Block Nagarabhavi ⇔ Shivajinagar...        244-C VSD   
1     5th Block HBR Layout ⇔ Kalyananagara Bus Stand          KLN-HBR   
2      5th Block HBR Layout ⇔ Kempegowda Bus Station              296   
3      5th Block HBR Layout ⇔ Kempegowda Bus Station            296-B   
4   5th Block HBR Layout ⇔ Shivajinagara Bus Station            296-D   

  agency_id route_type route_id  total_trips  total_stops  
0         1          3     3447          2.0         41.0  
1         1          3     7041          2.0          7.0  
2         1          3     2712          3.0         44.0  
3         1          3     4821          4.0         44.0  
4         1          3     2809          3.0         38.0  


In [3]:
def gtfs_hour(t):
    try:
        h, m, s = map(int, t.split(":"))
        return h + m / 60 + s / 3600
    except:
        return None

stop_times["arrival_hour"] = stop_times["arrival_time"].apply(gtfs_hour)
stop_times["departure_hour"] = stop_times["departure_time"].apply(gtfs_hour)

def time_period(h):
    if pd.isna(h):
        return "Unknown"
    if 5 <= h < 12:
        return "Morning"
    if 12 <= h < 17:
        return "Afternoon"
    if 17 <= h < 21:
        return "Evening"
    return "Night"

stop_times["arrival_period"] = stop_times["arrival_hour"].apply(time_period)
stop_times["departure_period"] = stop_times["departure_hour"].apply(time_period)

# Count each trip once, using its first departure
first_departures = stop_times.sort_values("stop_sequence").drop_duplicates("trip_id")

trip_hours = first_departures.merge(
    trips[["trip_id", "route_id"]],
    on="trip_id",
    how="left"
)

hourly_trips = (
    trip_hours.groupby(["route_id", "departure_hour"])["trip_id"]
    .nunique()
    .reset_index(name="trip_count")
)

print(stop_times[["arrival_hour", "departure_hour", "departure_period"]].head())
print("\nHourly trips:")
print(hourly_trips.head())

   arrival_hour  departure_hour departure_period
0     17.750000       17.750000          Evening
1     17.787500       17.787500          Evening
2     17.917778       17.917778          Evening
3     17.975278       17.975278          Evening
4     18.029444       18.029444          Evening

Hourly trips:
  route_id  departure_hour  trip_count
0     1000        4.916667           1
1     1000        5.166667           1
2     1000        5.333333           1
3     1000        5.666667           1
4     1000        6.083333           1


In [4]:
output_path = Path("../data/processed/bengaluru")
output_path.mkdir(parents=True, exist_ok=True)

route_features.to_csv(output_path / "route_profile.csv", index=False)
hourly_trips.to_csv(output_path / "hourly_trip_counts.csv", index=False)
stop_times.to_csv(output_path / "time_features.csv", index=False)

print("Bengaluru CSV files saved successfully!")

Bengaluru CSV files saved successfully!


In [5]:
print("Missing values:\n", route_features.isnull().sum())
print("\nDuplicate route IDs:", route_features["route_id"].duplicated().sum())
print("Duplicate trip IDs:", trips["trip_id"].duplicated().sum())
print("Invalid arrival hours:", stop_times["arrival_hour"].isna().sum())
print("Invalid departure hours:", stop_times["departure_hour"].isna().sum())

Missing values:
 route_long_name      0
route_short_name     0
agency_id            0
route_type           0
route_id             0
total_trips         18
total_stops         18
dtype: int64

Duplicate route IDs: 0
Duplicate trip IDs: 0
Invalid arrival hours: 0
Invalid departure hours: 0


In [6]:
def gtfs_minute(t):
    try:
        h, m, s = map(int, t.split(":"))
        return m
    except:
        return None

stop_times["arrival_minute"] = stop_times["arrival_time"].apply(gtfs_minute)
stop_times["departure_minute"] = stop_times["departure_time"].apply(gtfs_minute)

stop_times["stop_sequence"] = pd.to_numeric(
    stop_times["stop_sequence"], errors="coerce"
)

first_departures = (
    stop_times.sort_values("stop_sequence")
    .drop_duplicates("trip_id")
)

first_departures["departure_service_hour"] = (
    first_departures["departure_hour"].apply(
        lambda h: int(h) if pd.notna(h) else None
    )
)

trip_hours = first_departures.merge(
    trips[["trip_id", "route_id"]],
    on="trip_id",
    how="left"
)

hourly_trips = (
    trip_hours.groupby(["route_id", "departure_service_hour"])["trip_id"]
    .nunique()
    .reset_index(name="trip_count")
)

output_path = Path("../data/processed/bengaluru")
hourly_trips.to_csv(output_path / "hourly_trip_counts.csv", index=False)
stop_times.to_csv(output_path / "time_features.csv", index=False)

print("Bengaluru updated successfully!")

Bengaluru updated successfully!
